# Importation des bibliothèques

In [ ]:
import os
import sys
from pathlib import Path

import findspark
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat,
    initcap,
    length,
    lit,
    lower,
    regexp_replace,
    to_date,
    trim,
    when,
)

In [ ]:
# Configuration des variables d'environnement pour Java, Spark et Hadoop
os.environ["JAVA_HOME"] = (
    r"C:\Users\ahoun\AppData\Local\Programs\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
)
os.environ["SPARK_HOME"] = r"C:\spark\spark-3.5.6-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

findspark.init()

# Constantes

In [ ]:
# chemin pour accéder aux données
FILE_PATH = "../src/data/raw/donnees_immobilieres.parquet"

# chemin pour sauvegarder les données nettoyées
OUTPUT_PATH = "../src/data/clean/donnees_immobilieres_cleaned.delta"

# Application

In [ ]:
# Init SparkSession avec Delta Lake
builder = (
    SparkSession.builder.appName("RealStateProcessing")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.sql.warehouse.dir", "C:/tmp/spark-warehouse")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
# chargement des données
df_raw = spark.read.parquet(FILE_PATH)

In [ ]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df_raw.count(), len(df_raw.columns))

In [ ]:
# Affichage du schéma
df_raw.printSchema()

In [ ]:
# Affichage des 5 premières lignes
df_raw.show(5)

In [ ]:
# suppression des doublons suivant l'id
df = df_raw.dropDuplicates(["id"])

In [ ]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

In [ ]:
# Vérification qu'il n'y a plus de doublons sur l'id
df.groupBy("id").count().filter(col("count") > 1).count()

In [ ]:
# filtrage pour ne garder que les biens en France
df = df.filter(col("pays") == "France")

In [ ]:
# affichage des pays distincts
df.select("pays").distinct().show()

In [ ]:
# conversion de la colonne "date_inventaire" en type date
df = df.withColumn("date_inventaire", to_date(col("date_inventaire"), "yyyy-MM"))

In [ ]:
# Affichage du schéma
df.printSchema()

In [ ]:
# mise en minuscule de la colonne "ville" pour uniformiser les données
df = df.withColumn("ville", lower(col("ville")))

In [ ]:
# affichage des villes distinctes
df.select("ville").distinct().show()

In [ ]:
# mise en minuscule de la colonne "region" pour uniformiser les données
df = df.withColumn("region", lower(col("region")))

In [ ]:
# affichage des régions distinctes
df.select("region").distinct().show()

In [ ]:
# anonymisation des données sensibles
df = df.withColumn(
    "ville",
    when(
        col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle"
    ).otherwise(col("ville")),
)

In [ ]:
# affichage des villes distinctes
df.select("ville").distinct().show()

In [ ]:
# anonymisation des données sensibles
df = df.withColumn(
    "code_postal",
    when(
        col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle"
    ).otherwise(col("code_postal")),
)

In [ ]:
# affichage des codes postaux distincts
df.select("code_postal").distinct().show()

In [ ]:
# anonymisation des données sensibles
df = df.withColumn(
    "adresse",
    when(
        col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle"
    ).otherwise(col("adresse")),
)

In [ ]:
# affichage des adresses distinctes
df.select("adresse").distinct().show()

In [ ]:
# Affichage du nombre de départements distincts
df.select("dept").distinct().count()

In [ ]:
# Unification des codes départements : ajout d'un zéro devant les codes à un chiffre
df = df.withColumn(
    "dept",
    when(length(col("dept")) == 1, concat(lit("0"), col("dept")))
    .when(col("dept") == "N/A", None)
    .otherwise(col("dept")),
)

In [ ]:
# Affichage du nombre de départements distincts
df.select("dept").distinct().count()

In [ ]:
# affichage du nombre de types distincts
df.select("type").distinct().count()

In [ ]:
# Unification des types en catégories plus larges
df = df.withColumn(
    "type",
    when(
        lower(col("type")).rlike("ouvrage.*art|ouvrage.*réseau"),
        "Ouvrage d'art des réseaux et voiries",
    )
    .when(lower(col("type")).rlike("réseau|voirie"), "Réseaux et voirie")
    .when(
        lower(col("type")).rlike("enseignement|sport"),
        "Bâtiment d'enseignement ou de sport",
    )
    .when(lower(col("type")).contains("culte"), "Édifice du culte")
    .when(lower(col("type")).rlike("sanitaire|social"), "Bâtiment sanitaire ou social")
    .when(
        lower(col("type")).rlike("agricole|élevage"), "Bâtiment agricole ou d'élevage"
    )
    .when(lower(col("type")).contains("bureau"), "Bureau")
    .otherwise(initcap(regexp_replace(trim(lower(col("type"))), r"[.,;]+$", ""))),
)

In [ ]:
# affichage du nombre de types distincts
df.select("type").distinct().count()

In [ ]:
# affichage du nombre de ministères distincts
df.select("ministere").distinct().count()

In [ ]:
# nettoyage de la colonne "ministere" : suppression des espaces, mise en minuscule et suppression des ponctuations finales
df = df.withColumn(
    "ministere", regexp_replace(trim(lower(col("ministere"))), r"[.,;]+$", "")
)

In [ ]:
# uniformisation des données du ministère
df = df.withColumn(
    "ministere",
    when(
        lower(col("ministere")).rlike("éducation|education|enseignement sup"),
        "Ministère de l'Éducation Nationale",
    )
    .when(lower(col("ministere")).contains("justice"), "Ministère de la Justice")
    .when(
        lower(col("ministere")).rlike(
            "écologie|ecologie|environnement|aménagement|dura"
        ),
        "Ministère de l'Écologie",
    )
    .when(
        lower(col("ministere")).rlike("économie|economie|finances|industrie"),
        "Ministère de l'Économie et des Finances",
    )
    .when(
        lower(col("ministere")).rlike("budget|comptes publics|fonction publique"),
        "Ministère du Budget",
    )
    .when(
        lower(col("ministere")).rlike("affaires étrangères|etrangeres|européennes"),
        "Ministère des Affaires Étrangères",
    )
    .when(lower(col("ministere")).contains("intérieur"), "Ministère de l'Intérieur")
    .when(
        lower(col("ministere")).rlike("agriculture|pêche"), "Ministère de l'Agriculture"
    )
    .when(lower(col("ministere")).contains("culture"), "Ministère de la Culture")
    .when(lower(col("ministere")).contains("santé"), "Ministère de la Santé")
    .when(
        lower(col("ministere")).rlike("travail|solidarité|relations soc"),
        "Ministère du Travail",
    )
    .when(
        lower(col("ministere")).rlike("premier ministre|services du premier"),
        "Services du Premier Ministre",
    )
    .when(lower(col("ministere")).rlike("multi.*occup"), "Multi-occupants")
    .otherwise(initcap(regexp_replace(trim(lower(col("ministere"))), r"[.,;]+$", ""))),
)

In [ ]:
df.select("ministere").distinct().count()

In [ ]:
# affichage des ministères distincts
df.select("ministere").distinct().show()

In [ ]:
# Affichage du schéma
df.printSchema()

In [ ]:
# Affichage des 5 premières lignes
df.show(5)

In [ ]:
# Informations générales sur le DataFrame
df_raw.describe().show()

In [ ]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

In [ ]:
# Création du dossier si il n'existe pas
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Sauvegarde des données nettoyées au format delta
df.write.format("delta").mode("overwrite").save(OUTPUT_PATH)